
# ⚽ FIFA World Cup 2026 — Monte Carlo Intelligence Engine

## A Kaggle Grandmaster-Style Storytelling Notebook

This notebook goes beyond EDA:
- 🎲 10,000 Tournament Monte Carlo Simulations
- 🏆 Trophy Probability Rankings
- 🐎 Dark Horse Detection
- ⚡ Upset Probability Matrix
- 🌍 Confederation Intelligence
- 🗺️ Interactive World Maps
- 🌊 Tournament Path Sankey Diagrams
- 📈 ELO Storytelling
- 🎯 Professional Dark Dashboards

Designed to maximize engagement and discussion.


In [ ]:

import pandas as pd, numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default='notebook_connected'

df = pd.read_csv('/kaggle/input/your-dataset/elo_ratings_wc2026.csv')
df.head()


## Dataset Audit

In [ ]:

print(df.shape)
display(df.head())
display(df.describe(include='all'))


## 🌍 Global Rankings

In [ ]:

num = df.select_dtypes(include='number').columns
rating_col = num[-1]

top = df.sort_values(rating_col, ascending=False).head(25)

fig = px.bar(
    top,
    x=top.columns[0],
    y=rating_col,
    color=rating_col,
    template='plotly_dark',
    title='Top 25 Teams by ELO'
)
fig.show()


## 🎲 Monte Carlo Tournament Engine (10,000 Simulations)

In [ ]:

rating_col = df.select_dtypes(include='number').columns[-1]
team_col = df.columns[0]

ratings = dict(zip(df[team_col], df[rating_col]))

def win_prob(a,b):
    return 1/(1+10**((b-a)/400))

wins = {t:0 for t in ratings}

teams=list(ratings.keys())

for _ in range(10000):
    remaining=teams.copy()
    np.random.shuffle(remaining)

    while len(remaining)>1:
        nxt=[]
        for i in range(0,len(remaining)-1,2):
            t1,t2=remaining[i],remaining[i+1]
            p=win_prob(ratings[t1],ratings[t2])
            nxt.append(t1 if np.random.rand()<p else t2)

        if len(remaining)%2==1:
            nxt.append(remaining[-1])

        remaining=nxt

    wins[remaining[0]]+=1

mc=pd.DataFrame({
'Team':wins.keys(),
'Titles':wins.values()
})

mc['Probability']=mc['Titles']/10000

mc.sort_values('Probability',ascending=False).head(20)


In [ ]:

fig = px.bar(
    mc.sort_values('Probability',ascending=False).head(20),
    x='Team',
    y='Probability',
    color='Probability',
    template='plotly_dark',
    title='🏆 World Cup Trophy Probabilities'
)
fig.show()


## 🐎 Dark Horse Detector

In [ ]:

elite_cutoff = mc['Probability'].quantile(.9)

dark_horses = mc[
(mc['Probability'] < elite_cutoff) &
(mc['Probability'] > mc['Probability'].median())
]

dark_horses.sort_values('Probability',ascending=False)


## ⚡ Upset Probability Matrix

In [ ]:

top16 = df.sort_values(rating_col,ascending=False).head(16)

matrix=[]

for _,r1 in top16.iterrows():
    row=[]
    for _,r2 in top16.iterrows():
        p=1/(1+10**((r2[rating_col]-r1[rating_col])/400))
        row.append(round(1-p,3))
    matrix.append(row)

heat=pd.DataFrame(
matrix,
index=top16.iloc[:,0],
columns=top16.iloc[:,0]
)

fig = px.imshow(
heat,
template='plotly_dark',
title='Upset Probability Matrix'
)
fig.show()


## 🌍 Confederation Intelligence

In [ ]:

# If confederation column exists, this section activates.
possible=[c for c in df.columns if 'confed' in c.lower()]

if len(possible):
    conf=possible[0]
    fig=px.box(
        df,
        x=conf,
        y=rating_col,
        color=conf,
        template='plotly_dark',
        title='Confederation Strength'
    )
    fig.show()
else:
    print('No confederation column found.')


## 🗺️ Interactive World Map

In [ ]:

country_col=df.columns[0]

fig=px.choropleth(
    df,
    locations=country_col,
    locationmode='country names',
    color=rating_col,
    template='plotly_dark',
    title='Global ELO Landscape'
)

fig.show()


## 🌊 Tournament Path Sankey Prototype

In [ ]:

labels=['Group','R16','QF','SF','Final','Champion']

fig=go.Figure(go.Sankey(
node=dict(label=labels),
link=dict(
source=[0,1,2,3,4],
target=[1,2,3,4,5],
value=[32,16,8,4,1]
)
))

fig.update_layout(
template='plotly_dark',
title='World Cup Elimination Flow'
)

fig.show()


## 🚀 Executive Dashboard

In [ ]:

fig=make_subplots(
rows=2,cols=2,
subplot_titles=(
'ELO Distribution',
'Top Contenders',
'Trophy Probabilities',
'Ranking Curve'
)
)

fig.add_trace(
go.Histogram(x=df[rating_col]),
row=1,col=1
)

top=df.sort_values(rating_col,ascending=False).head(10)

fig.add_trace(
go.Bar(x=top.iloc[:,0],y=top[rating_col]),
row=1,col=2
)

best=mc.sort_values('Probability',ascending=False).head(10)

fig.add_trace(
go.Bar(x=best['Team'],y=best['Probability']),
row=2,col=1
)

rank=df.sort_values(rating_col,ascending=False)

fig.add_trace(
go.Scatter(y=rank[rating_col],mode='lines'),
row=2,col=2
)

fig.update_layout(
height=950,
template='plotly_dark',
title='⚽ World Cup 2026 Intelligence Dashboard'
)

fig.show()



# 🧠 Narrative Conclusions

### Title Favorites
Use Monte Carlo probabilities instead of raw ELO rankings.

### Dark Horses
Mid-tier teams with surprisingly high title odds deserve attention.

### Confederation Power
Measure depth, not just the strongest team.

### Upset Potential
World Cups are often decided by a handful of low-probability matches.

### Future Extensions
- Full bracket simulator
- Match-level forecasting
- Bayesian updating after each match
- SHAP explanations for predictive models
- Live World Cup dashboard

⭐ If this notebook taught you something new, consider an upvote.
